In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# =========================================================
# 1. CHUẨN BỊ ĐẶC TRƯNG ĐỒNG BỘ PIPELINE KPI & K-MEANS
# =========================================================
# Giả định 'zone_features_ml' là bảng tổng hợp các khu vực bao gồm cả nhãn Cụm và KPI nâng cao
# Nếu chưa có, ta tạo biến target: revenue_per_trip = revenue_total / trips
if 'revenue_per_trip' not in zone_features.columns:
    zone_features['revenue_per_trip'] = zone_features['revenue_total'] / zone_features['trips']

# Lọc bỏ các vùng không có dữ liệu hoặc có lượng chuyến quá thấp để tránh nhiễu mô hình ML
ml_data = zone_features[zone_features['trips'] > 10].copy()

# Lựa chọn các đặc trưng cao cấp đã được tính toán ở pipeline KPI trước đó
feature_cols = [
    'trips', 
    'duration_p50', 
    'duration_p95', 
    'speed_p50', 
    'tip_percent_median', 
    'flex_fare_percent', 
    'airport_percent'
]

X_numerical = ml_data[feature_cols]

# One-Hot Encoding cho biến định danh 'cluster' để thuật toán Ridge không bị hiểu sai logic
X_categorical = pd.get_dummies(ml_data['cluster'], prefix='cluster', drop_first=False)

# Gộp tất cả các đặc trưng sạch lại làm một
X = pd.concat([X_numerical, X_categorical], axis=1)
y = ml_data['revenue_per_trip']

# Chia tập dữ liệu Train/Test theo tỷ lệ chuẩn 80/20
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"📊 Kích thước tập dữ liệu huấn luyện ML: {X_train.shape}")
print(f"🧬 Các đặc trưng đưa vào mô hình: \n{list(X.columns)}\n")

# =========================================================
# 2. HỒI QUY ĐIỀU CHUẨN (Ridge) - TÍCH HỢP CHUẨN HÓA MẠNH MẼ
# =========================================================
# Ridge bắt buộc phải có StandardScaler để đưa các biến về cùng một hệ quy chiếu
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

ridge_model = Ridge(alpha=1.0)
ridge_model.fit(X_train_scaled, y_train)
y_pred_ridge = ridge_model.predict(X_test_scaled)

# =========================================================
# 3. ENSEMBLE LEARNING (Random Forest Regressor)
# =========================================================
rf_model = RandomForestRegressor(n_estimators=150, max_depth=10, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train) # Tree-based không cần scaled dữ liệu
y_pred_rf = rf_model.predict(X_test)

# =========================================================
# 4. SO SÁNH ĐÁNH GIÁ CHUYÊN SÂU DÀNH CHO BÁO CÁO
# =========================================================
def evaluate_model_performance(y_true, y_pred, model_name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f"=== KẾT QUẢ MÔ HÌNH {model_name.upper()} ===")
    print(f"  - RMSE (Sai số căn phương): {rmse:.2f} USD/chuyến")
    print(f"  - MAE  (Sai số tuyệt đối):  {mae:.2f} USD/chuyến")
    print(f"  - R²   (Hệ số xác định):    {r2*100:.2f}%\n")
    return r2

print("="*50)
r2_ridge = evaluate_model_performance(y_test, y_pred_ridge, "Ridge Regression")
r2_rf = evaluate_model_performance(y_test, y_pred_rf, "Random Forest")
print("="*50)

# =========================================================
# 5. TRỰC QUAN HÓA ĐỘ QUAN TRỌNG ĐẶC TRƯNG CHUẨN HÓA
# =========================================================
features_list = X.columns
importances = rf_model.feature_importances_
indices = np.argsort(importances)

plt.figure(figsize=(10, 6))
plt.title('Độ Quan Trọng Của Các Đặc Trưng Trong Việc Tối Ưu Doanh Thu', fontsize=12, fontweight='bold', pad=15)
plt.barh(range(len(indices)), importances[indices], color='#2ca02c', align='center', alpha=0.85)
plt.yticks(range(len(indices)), [features_list[i] for i in indices], fontsize=10)
plt.xlabel('Mức độ đóng góp giải thích biến động (Relative Importance)', fontsize=10)
plt.grid(axis='x', linestyle=':', alpha=0.5)
plt.tight_layout()

# Lưu ảnh phục vụ đồ án
plt.savefig("../../Data/figures/advanced_pattern/revenue_feature_importance.png", dpi=300)
plt.show()